# Notebook para Análise de Dados com PokeAPI

- **Objetivo**.

Nesta etapa, o(a) candidato(a) deverá consumir dados diretamente da API pública da PokeAPI e
realizar análises utilizando Apache Spark.O objetivo é avaliar habilidades de:

    - Ingestão de dados a partir de API REST
    - Tratamento de paginação e dados aninhados (JSON)
    - Modelagem relacional
    - Manipulação e agregação de dados com Spark
    - Clareza na organização e documentação do código

In [8]:
import requests
import asyncio
import aiohttp
from aiohttp import ClientSession
from typing import List, Dict, Any
import json

BASE_URL = "https://pokeapi.co/api/v2/pokemon/"
CONCURRENT_REQUESTS_LIMIT = 20 

concurrent_limit = asyncio.Semaphore(CONCURRENT_REQUESTS_LIMIT)

## Etapa 1 — Data Extraction

Consumir o endpoint /pokemon para obter a listagem completa de todos os pokémons.

1. Para cada url retornada, realizar uma nova requisição e extrair exclusivamente os seguintes
campos e transformá los em novas tabelas:
- types
- stats
- abilities

In [ ]:
async def fetch_url_json(session: aiohttp.ClientSession, url: str, semaphore: asyncio.Semaphore) -> Dict[str, Any]:
    async with semaphore:
        try:
            async with session.get(url) as response:
                response.raise_for_status()
                return await response.json()
        except (aiohttp.ClientError, asyncio.TimeoutError) as e:
            print(f"Error fetching {url}: {e}")
            return {}

async def fetch_all_pokemons_urls() -> List[Dict[str, Any]]:
    timeout = aiohttp.ClientTimeout(total=30)
    all_pokemon_data = []
    async with aiohttp.ClientSession(timeout=timeout) as session:
        data = await fetch_url_json(session, BASE_URL, concurrent_limit)
        all_pokemon_data.extend(data["results"])
        while data.get("next"):
            next_url = data["next"]
            data = await fetch_url_json(session, next_url, concurrent_limit)
            all_pokemon_data.extend(data["results"])
    return all_pokemon_data

pokemon_full_data = await fetch_all_pokemons_urls()

with open("data/pokemon_urls_list.json", "w", encoding="utf-8") as f:
    json.dump(pokemon_full_data, f, indent=2, ensure_ascii=False)

print("OK, salvo com sucesso.")

OK, salvo com sucesso.


In [18]:
pokemon_full_data[:5]

[{'name': 'bulbasaur', 'url': 'https://pokeapi.co/api/v2/pokemon/1/'},
 {'name': 'ivysaur', 'url': 'https://pokeapi.co/api/v2/pokemon/2/'},
 {'name': 'venusaur', 'url': 'https://pokeapi.co/api/v2/pokemon/3/'},
 {'name': 'charmander', 'url': 'https://pokeapi.co/api/v2/pokemon/4/'},
 {'name': 'charmeleon', 'url': 'https://pokeapi.co/api/v2/pokemon/5/'}]

In [14]:
teste1 = ["a","b","c"]
teste2 = ["d","e","f"]

teste1.extend(teste2)
teste1

['a', 'b', 'c', 'd', 'e', 'f']

## Etapa 2 — Data Modeling

Ao final da extração, os dados deverão estar organizados de acordo com as tabelas apresentadas no
tópico Dicionario de Dados na seção Anexos no final desse documento.

## Etapa 3 — Data Analysis using Spark

Após a construção das tabelas, o(a) candidato(a) deverá responder às seguintes perguntas utilizando
Apache Spark:
Caso você não tenha um ambiente Spark, tente ver o Databricks Community Edition.